# iSD toy experiment

Compare four toy objectives on a 2D shifted-Gaussian transport problem:

1. **CFM only**
2. **Naive self-distillation only**
3. **CFM + naive self-distillation**
4. **iSD-style CFM + SD-R**

The naive objectives are kept as instability baselines. The final iSD-style model uses the `t=0: data`, `t=1: noise` convention and learns an off-diagonal flow-map representation.

## 1. Setup

In [ ]:
import torch
import matplotlib.pyplot as plt

from flowmodels.isd import (
    FlowMap,
    cfm_loss,
    self_distillation_loss,
    naive_isd_loss,
    ISDFlowMap,
    isd_loss,
)

torch.manual_seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
cfm_model = FlowMap().to(device)
sd_model = FlowMap().to(device)
naive_isd_model = FlowMap().to(device)
isd_model = ISDFlowMap().to(device)

## 2. CFM baseline

Train the original flow-map model using only local CFM velocity grounding.

In [ ]:
cfm_optimizer = torch.optim.Adam(cfm_model.parameters(), lr=1e-3)
cfm_history = []

for step in range(2000):
    loss = cfm_loss(cfm_model, batch_size=256, device=device)

    cfm_optimizer.zero_grad()
    loss.backward()
    cfm_optimizer.step()

    cfm_history.append(loss.item())

    if (step + 1) % 200 == 0:
        print(f"step {step+1:4d} | CFM loss: {loss.item():.4f}")

plt.figure(figsize=(6, 4))
plt.plot(cfm_history)
plt.xlabel("Training step")
plt.ylabel("CFM loss")
plt.title("CFM-only training")
plt.show()

## 3. Naive self-distillation baseline

This baseline only enforces agreement between a direct map and a two-step composed map. It has no CFM grounding and is included to expose the instability of this naive objective.

In [ ]:
sd_optimizer = torch.optim.Adam(sd_model.parameters(), lr=1e-3)
sd_history = []

for step in range(2000):
    loss = self_distillation_loss(sd_model, batch_size=256, device=device)

    sd_optimizer.zero_grad()
    loss.backward()
    sd_optimizer.step()

    sd_history.append(loss.item())

    if (step + 1) % 200 == 0:
        print(f"step {step+1:4d} | SD loss: {loss.item():.6f}")

plt.figure(figsize=(6, 4))
plt.plot(sd_history)
plt.xlabel("Training step")
plt.ylabel("SD loss")
plt.title("Naive SD-only training")
plt.show()

## 4. CFM + naive self-distillation

Add CFM grounding to the same naive two-step self-distillation objective. This is still a comparison baseline, not the final SD-R objective.

In [ ]:
naive_optimizer = torch.optim.Adam(naive_isd_model.parameters(), lr=1e-3)
naive_total_history = []
naive_cfm_history = []
naive_sd_history = []

for step in range(2000):
    loss, loss_cfm, loss_sd = naive_isd_loss(
        naive_isd_model,
        batch_size=256,
        lambda_sd=1.0,
        device=device,
    )

    naive_optimizer.zero_grad()
    loss.backward()
    naive_optimizer.step()

    naive_total_history.append(loss.item())
    naive_cfm_history.append(loss_cfm.item())
    naive_sd_history.append(loss_sd.item())

    if (step + 1) % 200 == 0:
        print(
            f"step {step+1:4d} | "
            f"total: {loss.item():.4f} | "
            f"CFM: {loss_cfm.item():.4f} | "
            f"SD: {loss_sd.item():.4f}"
        )

plt.figure(figsize=(7, 4))
plt.plot(naive_total_history, label="Total")
plt.plot(naive_cfm_history, label="CFM")
plt.plot(naive_sd_history, label="Naive SD")
plt.xlabel("Training step")
plt.ylabel("Loss")
plt.title("CFM + naive SD")
plt.legend()
plt.show()

## 5. iSD-style training: CFM + SD-R

The final model uses the reversed convention `t=0: data`, `t=1: noise`. CFM grounds the diagonal velocity prediction, while SD-R trains the off-diagonal flow-map representation with a stop-gradient regression target.

In [ ]:
isd_optimizer = torch.optim.Adam(isd_model.parameters(), lr=1e-3)
isd_total_history = []
isd_cfm_history = []
isd_sdr_history = []

for step in range(2000):
    loss, loss_cfm, loss_sd = isd_loss(
        isd_model,
        batch_size=256,
        lambda_sd=1.0,
        device=device,
    )

    isd_optimizer.zero_grad()
    loss.backward()
    isd_optimizer.step()

    isd_total_history.append(loss.item())
    isd_cfm_history.append(loss_cfm.item())
    isd_sdr_history.append(loss_sd.item())

    if (step + 1) % 200 == 0:
        print(
            f"step {step+1:4d} | "
            f"total: {loss.item():.4f} | "
            f"CFM: {loss_cfm.item():.4f} | "
            f"SD-R: {loss_sd.item():.6f}"
        )

plt.figure(figsize=(7, 4))
plt.plot(isd_total_history, label="Total")
plt.plot(isd_cfm_history, label="CFM")
plt.plot(isd_sdr_history, label="SD-R")
plt.xlabel("Training step")
plt.ylabel("Loss")
plt.title("iSD: CFM + SD-R")
plt.legend()
plt.show()

## 6. One-step generation

Generate directly from noise at `t=1` to data at `s=0`. `ISDFlowMap` predicts the flow-map vector, so the mapped position is recovered as `x_s = x_t + (s - t) F_theta(x_t; t, s)`.

In [ ]:
n = 2000

x_noise = torch.randn(n, 2, device=device)
t = torch.ones(n, 1, device=device)
s = torch.zeros(n, 1, device=device)

with torch.no_grad():
    F = isd_model(x_noise, t, s)
    x_generated = x_noise + (s - t) * F

x_target = torch.randn(n, 2, device=device)
x_target[:, 0] += 4.0

x_noise = x_noise.cpu()
x_generated = x_generated.cpu()
x_target = x_target.cpu()

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))

for ax, (x, title) in zip(
    axes,
    [
        (x_noise, "Noise (t=1)"),
        (x_target, "Target data (t=0)"),
        (x_generated, "iSD one-step"),
    ],
):
    ax.scatter(x[:, 0], x[:, 1], s=5, alpha=0.5)
    ax.set_title(title)
    ax.set_xlim(-4, 8)
    ax.set_ylim(-4, 4)
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()

for name, x in [("Target", x_target), ("iSD", x_generated)]:
    print(f"\n{name}")
    print("mean:", x.mean(dim=0))
    print("std :", x.std(dim=0))

## 7. Off-diagonal flow-map visualization

Query the same starting noise directly at several target times. These are independent `t=1 -> s` flow-map evaluations rather than a sequential numerical integration.

In [ ]:
n = 2000
x_noise = torch.randn(n, 2, device=device)
t = torch.ones(n, 1, device=device)
times = [0.75, 0.5, 0.25, 0.0]

fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))

x_plot = x_noise.cpu()
axes[0].scatter(x_plot[:, 0], x_plot[:, 1], s=5, alpha=0.5)
axes[0].set_title("t = 1.00")

for ax, s_value in zip(axes[1:], times):
    s = torch.full((n, 1), s_value, device=device)

    with torch.no_grad():
        F = isd_model(x_noise, t, s)
        x_s = x_noise + (s - t) * F

    x_s = x_s.cpu()
    ax.scatter(x_s[:, 0], x_s[:, 1], s=5, alpha=0.5)
    ax.set_title(f"t = {s_value:.2f}")

for ax in axes:
    ax.set_xlim(-4, 8)
    ax.set_ylim(-4, 4)
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()